<h1><center>Adult income based classification benchmark<h1>

In [17]:
# The classification dataset with 2 classes, the target value is annual income above 50K or below 50K.
# Source of the dataset: https://archive.ics.uci.edu/dataset/2/adult

In [18]:
# The final weight estimate, who match this exact person's age, race, and sex.

In [19]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

import sklearn
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

<h3><center>Loading and exploring the dataset<h3>

In [39]:
# I need to define name for columns, because the dataset has no header.
column_names = ["age", "workclass", "final_weight", "education", "education_num",
                "marital_status", "occupation", "relationship", "race", "sex",
                "capital_gain", "capital_loss", "hours_per_week", "native_country", "income"]

# What I noticed, the dataset has ? mark so  I am gonna convert the ? mark into nan. 
df = pd.read_csv("Adult_inccome/adult.data", header=None,
                names=column_names, skipinitialspace=True, na_values="?")

print(f"Dataset shape: {df.shape}")

Dataset shape: (32561, 15)


In [38]:
df.head()

,age,workclass,inal_weight,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


In [22]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 32561 entries, 0 to 32560
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   age             32561 non-null  int64
 1   workclass       30725 non-null  str  
 2   inal_weight     32561 non-null  int64
 3   education       32561 non-null  str  
 4   education_num   32561 non-null  int64
 5   marital_status  32561 non-null  str  
 6   occupation      30718 non-null  str  
 7   relationship    32561 non-null  str  
 8   race            32561 non-null  str  
 9   sex             32561 non-null  str  
 10  capital_gain    32561 non-null  int64
 11  capital_loss    32561 non-null  int64
 12  hours_per_week  32561 non-null  int64
 13  native_country  31978 non-null  str  
 14  income          32561 non-null  str  
dtypes: int64(6), str(9)
memory usage: 3.7 MB


In [23]:
# The DS has 6 numeric columns and 8 categorical columns. 
# The numeric_columns are: age, final_weight, education_num, capital_gain, capital_loss, hours_per_week
# The categorical_columns are: workclass, education, marital_status, occupation, relationship, race, sex, native_country

In [ ]:
# Missing values
missing_values = df.isna().sum()
print(f"\nMissing values", missing_values[missing_values > 0])

Duplicate rows: 24

Missing values workclass         1836
occupation        1843
native_country     583
dtype: int64


In [25]:
# let's check the annual income classes. 
In_come_info  = pd.DataFrame({'count': df['income'].value_counts(),'percent': df['income']
                              .value_counts(normalize=True).mul(100).round(2)})
In_come_info

,count,percent
income,,
<=50K,24720,75.92
>50K,7841,24.08


In [ ]:
# Which columns are missing together?

missing_columns = ['workclass','occupation','native_country']
print('Total rows with missing value:',df[missing_columns].isna().any(axis=1).sum())
missing_pattern = (df[missing_columns].isna().value_counts().rename("rows"))

missing_pattern
# There are 4262 missing values and 2399 rows have them. 
# Every row missing workclass is also missing occupation: 1809 + 27 = 1836 and only 7 rows have missing occupation.  

Total rows with missing value: 2399


workclass  occupation  native_country
False      False       False             30162
True       True        False              1809
False      False       True                556
True       True        True                 27
False      True        False                 7
Name: rows, dtype: int64

In [ ]:
# Compring missing values with target values: 
missing_data = (df[missing_columns].isna().any(axis=1))
print(pd.crosstab(missing_data,df["income"],margins=True))
# Most od missing values are associated with under 50K
# If I run df.dropna(), it would remove 2399 rows, around 7.3% of the data and change the class distribution overall. 
# mmm, I am gonna keep them with missing values, later I will use constant value  inside the categorica preprocessing pipeline. 

income  <=50K  >50K    All
row_0                     
False   22654  7508  30162
True     2066   333   2399
All     24720  7841  32561


In [47]:
# Let's check duplicated 
duplicate_rows = df[df.duplicated(keep=False)]
print("Rows in duplicate groups:",len(duplicate_rows))
print(duplicate_rows["income"].value_counts())

Rows in duplicate groups: 47
income
<=50K    43
>50K      4
Name: count, dtype: int64


In [ ]:
# Let's check education redundancy
education_mapping = (df[["education", "education_num"]].drop_duplicates().sort_values("education_num"))

print(education_mapping)

education_to_num = df.groupby("education")["education_num"].nunique()
num_to_education = df.groupby("education_num")["education"].nunique()

print("\nEducation categories with multiple numbers:",(education_to_num > 1).sum())

print("Education numbers with multiple categories:",(num_to_education > 1).sum())

        education  education_num
224     Preschool              1
160       1st-4th              2
56        5th-6th              3
15        7th-8th              4
6             9th              5
77           10th              6
3            11th              7
415          12th              8
2         HS-grad              9
10   Some-college             10
14      Assoc-voc             11
13     Assoc-acdm             12
0       Bachelors             13
5         Masters             14
52    Prof-school             15
20      Doctorate             16

Education categories with multiple numbers: 0
Education numbers with multiple categories: 0


In [ ]:
# Numeric feature structure
numeric_columns = ["age", "final_weight", "education_num",
                   "capital_gain", "capital_loss", "hours_per_week"]
print(df[numeric_columns].describe().T)

                  count           mean            std      min       25%  \
age             32561.0      38.581647      13.640433     17.0      28.0   
final_weight    32561.0  189778.366512  105549.977697  12285.0  117827.0   
education_num   32561.0      10.080679       2.572720      1.0       9.0   
capital_gain    32561.0    1077.648844    7385.292085      0.0       0.0   
capital_loss    32561.0      87.303830     402.960219      0.0       0.0   
hours_per_week  32561.0      40.437456      12.347429      1.0      40.0   

                     50%       75%        max  
age                 37.0      48.0       90.0  
final_weight    178356.0  237051.0  1484705.0  
education_num       10.0      12.0       16.0  
capital_gain         0.0       0.0    99999.0  
capital_loss         0.0       0.0     4356.0  
hours_per_week      40.0      45.0       99.0  

Zero values percentage:
capital_gain    91.67
capital_loss    95.33
dtype: float64


In [46]:
zero_percent = (df[["capital_gain", "capital_loss"]].eq(0).mean().mul(100).round(2))

print("\nZero values percentage:")
print(zero_percent)


Zero values percentage:
capital_gain    91.67
capital_loss    95.33
dtype: float64
